# Exemplo: Classificação binária
--------------------------------

Este exemplo mostra como usar o ExperionML para resolver um problema de classificação binária. Além disso, vamos executar diversas etapas de limpeza de dados para preparar os dados para a modelagem.

Os dados utilizados são uma variação do [conjunto de dados de clima australiano](https://www.kaggle.com/jsphyg/weather-dataset-rattle-package) do Kaggle. Você pode baixá-lo [aqui](https://github.com/tvdboom/ExperionML/blob/master/examples/datasets/weatherAUS.csv). O objetivo deste conjunto é prever se vai chover amanhã treinando um classificador binário com a variável alvo `RainTomorrow`.

## Carregar os dados

In [1]:
# Import packages
import pandas as pd
from experionml import ExperionMLClassifier

In [2]:
# Carregue os dados
X = pd.read_csv("./datasets/weatherAUS.csv")

# Let's have a look
X.head()

,Location,MinTemp,MaxTemp,Rainfall,Evaporation,Sunshine,WindGustDir,WindGustSpeed,WindDir9am,WindDir3pm,...,Humidity9am,Humidity3pm,Pressure9am,Pressure3pm,Cloud9am,Cloud3pm,Temp9am,Temp3pm,RainToday,RainTomorrow
0,MelbourneAirport,18.0,26.9,21.4,7.0,8.9,SSE,41.0,W,SSE,...,95.0,54.0,1019.5,1017.0,8.0,5.0,18.5,26.0,Yes,0
1,Adelaide,17.2,23.4,0.0,NaN,NaN,S,41.0,S,WSW,...,59.0,36.0,1015.7,1015.7,NaN,NaN,17.7,21.9,No,0
2,Cairns,18.6,24.6,7.4,3.0,6.1,SSE,54.0,SSE,SE,...,78.0,57.0,1018.7,1016.6,3.0,3.0,20.8,24.1,Yes,0
3,Portland,13.6,16.8,4.2,1.2,0.0,ESE,39.0,ESE,ESE,...,76.0,74.0,1021.4,1020.5,7.0,8.0,15.6,16.0,Yes,1
4,Walpole,16.4,19.9,0.0,NaN,NaN,SE,44.0,SE,SE,...,78.0,70.0,1019.4,1018.9,NaN,NaN,17.4,18.1,No,0


## Executar o pipeline

In [3]:
# Chame o experionml usando apenas 5% do conjunto completo, para fins ilustrativos
experionml = ExperionMLClassifier(X, y="RainTomorrow", n_rows=0.05, n_jobs=8, verbose=2)

<< ================== ExperionML ================== >>

Configuração ==================== >>
Tarefa do algoritmo: Binary classification.
Processamento paralelo com 8 núcleos.

Estatísticas do conjunto de dados ==================== >>
Formato: (7109, 22)
Tamanho do conjunto de train: 5688
Tamanho do conjunto de test: 1421
-------------------------------------
Memória: 1.25 MB
Escalonado: False
Valores ausentes: 15994 (10.2%)
Atributos categóricos: 5 (23.8%)



In [4]:
# Impute missing values
experionml.impute(strat_num="median", strat_cat="drop", max_nan_rows=0.8)

Ajustando Imputer...


Imputando valores ausentes...
 --> Removendo 4 amostras por conterem mais de 16 valores ausentes.
 --> Removendo 867 amostras por conterem valores ausentes em colunas categóricas.
 --> Imputando 11 valores ausentes com median (12.0) na coluna MinTemp.
 --> Imputando 2 valores ausentes com median (22.7) na coluna MaxTemp.
 --> Imputando 2579 valores ausentes com median (4.8) na coluna Evaporation.
 --> Imputando 2895 valores ausentes com median (8.4) na coluna Sunshine.
 --> Imputando 49 valores ausentes com median (70.0) na coluna Humidity9am.
 --> Imputando 80 valores ausentes com median (52.0) na coluna Humidity3pm.
 --> Imputando 512 valores ausentes com median (1017.7) na coluna Pressure9am.
 --> Imputando 509 valores ausentes com median (1015.3) na coluna Pressure3pm.
 --> Imputando 2354 valores ausentes com median (5.0) na coluna Cloud9am.
 --> Imputando 2477 valores ausentes com median (5.0) na coluna Cloud3pm.
 --> Imputando 18 valores ausentes com median (16.7) na coluna Temp9

In [5]:
# Codifique as variáveis categóricas
experionml.encode(strategy="Target", max_onehot=10, infrequent_to_value=0.04)

Ajustando Encoder...


Codificando colunas categóricas...
 --> Aplicando Target-encoding à variável Location. Ela contém 1 classes.
   --> Tratando 6238 classes desconhecidas.
 --> Aplicando Target-encoding à variável WindGustDir. Ela contém 16 classes.
 --> Aplicando Target-encoding à variável WindDir9am. Ela contém 16 classes.
 --> Aplicando Target-encoding à variável WindDir3pm. Ela contém 16 classes.
 --> Aplicando Ordinal-encoding à variável RainToday. Ela contém 2 classes.


In [6]:
# Treine um modelo Extra-Trees e um Random Forest
experionml.run(models=["ET", "RF"], metric="f1", n_bootstrap=5)


Training ========================= >>
Models: ET, RF
Metric: f1


Results for ExtraTrees:
Fit ---------------------------------------------


Train evaluation --> f1: 1.0
Test evaluation --> f1: 0.5263
Time elapsed: 0.459s


Bootstrap ---------------------------------------
Evaluation --> f1: 0.5267 ± 0.0128
Time elapsed: 0.997s
-------------------------------------------------
Time: 1.457s


Results for RandomForest:
Fit ---------------------------------------------


Train evaluation --> f1: 1.0
Test evaluation --> f1: 0.559
Time elapsed: 0.380s


Bootstrap ---------------------------------------
Evaluation --> f1: 0.5462 ± 0.0197
Time elapsed: 1.400s
-------------------------------------------------
Time: 1.780s


Resultados finais ==================== >>
Tempo total: 3.252s
-------------------------------------
ExtraTrees   --> f1: 0.5267 ± 0.0128 ~
RandomForest --> f1: 0.5462 ± 0.0197 ~ !


## Analisar os resultados

In [7]:
# Vamos observar os resultados finais
experionml.results

,f1_train,f1_test,time_fit,f1_bootstrap,time_bootstrap,time
ET,0.849800,0.535400,0.459176,0.526719,0.997428,1.456604
RF,1.000000,0.559000,0.380063,0.546246,1.400195,1.780258


In [8]:
# Visualize os resultados do bootstrap
experionml.plot_results(title="RF vs ET performance")

In [9]:
# Exiba os resultados de algumas métricas comuns
experionml.evaluate()

,accuracy,ap,ba,f1,jaccard,mcc,precision,recall,auc
ET,0.829500,0.650900,0.688000,0.535400,0.365600,0.463200,0.724600,0.424600,0.842300
RF,0.827100,0.668900,0.703600,0.559000,0.387900,0.467500,0.681800,0.473700,0.844100


In [10]:
# O atributo winner chama o melhor modelo (experionml.winner == experionml.rf)
print(f"O vencedor é o modelo {experionml.winner.name}!!")

O vencedor é o modelo RF!!


In [11]:
# Visualize a distribuição das probabilidades previstas
experionml.winner.plot_probabilities()

In [12]:
# Compare how different metrics perform for different thresholds
experionml.winner.plot_threshold(metric=["f1", "accuracy", "ap"], steps=50)